In [ ]:
model

## LoRA
$$
W_{\text{new}} = W_0 + \Delta W = W_0 + BA
$$

- $W_0$ 是模型原参数, 在微调中将被冻结
- $A \in \mathbb{R}^{r \times in}$, $B \in \mathbb{R}^{out \times r}$, $r \ll \min(in, out).$

##### **超参数alpha**
通过超参数引入缩放 alpha/r, 通常情况下alpha选定就不在改变.

缩放的核心目的是当秩 r 改变时, 不必重新调整学习率, 超参数更稳定、更易迁移.

论文中将 alpha 固定为第一次尝试的r的值(例如最初使用 r=16 时设 alpha=16，那么缩放因子就是 16/16 = 1). 当把 r 改大(比如变成 32)时，缩放因子自动变为0.5, 削弱了LoRA分支的幅值; 同理改小 r 时缩放因子会变大. 这种自动补偿保持了训练初始阶段更新幅度的稳定，从而让学习率等超参数在 r 改变时依然可用.

所以 alpha 本质上控制 LoRA 最终施加到原始权重上的影响力强度. alpha 越大 → alpha/r 越大 → LoRA 的增量对模型输出的改变越剧烈.

实践中常将 alpha 设为 r 的 1 ~ 2 倍.



##### **需要注意的细节**
1. 通常在初始化时 $B$ 将被初始化为0矩阵, 这是为了在微调开始时LoRA不改变原模型行为.
2. $A$ 我们会给予很小的初始化值, 这是为了在微调开始时不希望更新幅度太大导致训练不稳定.
   - 假设我们对 $A$ 按照标准正态分布初始化 `torch.randn(r, in_features)` 在第一次更新权重时 $\Delta W = B_\text(new)A$(其中$B_{\text{new}} = 0 - \eta \cdot \frac{\partial L}{\partial B} = -\eta \cdot \frac{\partial L}{\partial y} (Ax)^T$) 此时由于 $A$ 里元素的量级在 1 左右, 因此第一步的更新量可能会非常大.
   - 通常使用 Kaiming 均匀分布初始化 $A$.
3. 我们通常也不会让 $A$ 初始化为0, $B$ 初始化为随机矩阵, 这会带来新的问题.
   - 计算图:
       - 在实际计算中 $\Delta_y=x \cdot A \cdot B$, 这里计算是有顺序的, 因此对于梯度将先得到 $\frac{\partial L}{\partial B} = \left( \frac{\partial L}{\partial \Delta y} \right)^T \cdot (xA^T)$ 之后得到 $\frac{\partial L}{\partial A} = B^T \cdot \left( \frac{\partial L}{\partial \Delta y} \right)^T \cdot x$.
       - 假设按照正常初始化, $\frac{\partial L}{\partial B}$ 结果是将预训练模型输出的干净误差信号($\frac{\partial L}{\partial \Delta y}$) 直接传到 B 上(因为 $Ax$ 很小相当于对信号给予了小的缩放因子, 无负面影响反而更稳定)。 $\frac{\partial L}{\partial A}$ 的结果中由于 $B=0$, 所以 $A$ 的梯度一开始是 0，暂时不动。等 $B$ 有了微小更新后，$A$ 才开始被优化，路径平滑.
       - 交换初始化, 此时 $\frac{\partial L}{\partial B}$ 为0, 对于 $\frac{\partial L}{\partial A}$ 则不同, 注意计算顺序, $B$ 先把干净的误差信号($\frac{\partial L}{\partial \Delta y}$)经过一次随机线性混合才传给 $A$, 相当于给优化过程注入了噪声.
   - 维度:
       - $A$ 是从高维输入到低秩空间的映射; $B$ 是低秩空间到高维输出空间的映射.
       - 在标准初始化中，$A$ 虽然随机，但它只是提供了一个低秩的基；$B$ 从零开始学习这个基的组合系数。这个任务相对容易，因为 $B$ 的输出空间维度虽大，但它要学的参数非常少.
       - 在互换初始化中，$B$ 先提供了随机但固定的输出基，$A$ 要从零开始学习如何把输入映射到这个预先指定的随机基上。这就变成了一个更难的问题：$A$ 必须学会在输入空间中找到一个投影，使得经过 $B$ 的组合后能产生正确的误差下降方向。而 $B$ 的随机性会导致这个目标函数在多步训练初期极不平滑，容易让 A 陷入较差的局部解.

In [ ]:
import os
import math
import torch
import inspect
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM

from train import Train
from data.data import get_dateiter

In [ ]:
class LoraLayer(nn.Module):
    def __init__(self, original_layer, r=8, alpha=16):
        super().__init__()
        self.original_layer = original_layer
        self.original_layer.weight.requires_grad = False
        if self.original_layer.bias is not None:
            self.original_layer.bias.requires_grad = False  # 原始线性层, 冻结weight和bias

        in_features, out_features = original_layer.in_features, original_layer.out_features

        self.lora_a = nn.Parameter(torch.empty(r, in_features, dtype=torch.bfloat16))
        nn.init.kaiming_uniform_(self.lora_a)
        self.lora_b = nn.Parameter(torch.zeros(out_features, r, dtype=torch.bfloat16))
        self.scaling = alpha / r

    def forward(self, X):
        # 不计算原始输出梯度
        with torch.no_grad():
            original_out = self.original_layer(X)
        lora_out = (X @ self.lora_a.T) @ self.lora_b.T * self.scaling

        return original_out + lora_out

In [ ]:
# 替换模型中所有Linear层为LoraLinear
def apply_lora_to_model(model, target_modules: list, r: int = 8, alpha: int = 16):
    """
    target_modules: 指定要替换的模块名列表，如 ['q_proj', 'v_proj']
    """
    def replace_module(model, parent=None, name=None):
        for child_name, child_model in list(model.named_children()):
            if isinstance(child_model, nn.Linear):
                full_name = f"{name}.{child_name}" if name else child_name
                should_replace = any(t in full_name for t in target_modules)

                if should_replace:
                    lora_layer = LoraLayer(child_model, r, alpha)
                    setattr(model, child_name, lora_layer)
            else:
                new_name = f"{name}.{child_name}" if name else child_name
                replace_module(child_model, parent=model, name=new_name)
                    
    replace_module(model)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    return trainable_params

In [ ]:
batch_size = 5
lr = 2e-4
lora_r = 8
lora_alpha = 16
num_workers = os.cpu_count() // 2

train_iter = get_dateiter(batch_size, 192, num_workers, is_train=True) # 数据最长为192token
net = AutoModelForCausalLM.from_pretrained(
        "/root/autodl-tmp/SFTwithLoRA/LLM",
        dtype=torch.bfloat16,
        trust_remote_code=True
    )

target_layer = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
trainable_params = apply_lora_to_model(net, target_layer, r=lora_r, alpha=lora_alpha)

loss = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)  # AdamW区别于Adam, 对权重衰减进行了解耦

In [ ]:
# python -m torch.distributed.launch --nproc_per_node=NUM_GPUS train.py
num_epochs = 5

class TrainLoRA(Train):
    def train_epochs(self, num_epochs, train_iter, test_iter):
        for epoch in range(num_epochs):
            self.net.train()
            for batch_idx, batch in enumerate(train_iter):
                self.optimizer.zero_grad(set_to_none=True)
                X, Y, attention_mask = [x.to(self.device) for x in batch]
                outputs = self.net(input_ids=X, attention_mask=attention_mask)
                logits = outputs.logits
                logits_flat = logits.view(-1, logits.size(-1))
                labels_flat = Y.view(-1)
                l = self.loss(logits.view(-1, logits.size(-1)), Y.view(-1))
                l.backward()
                
                params = [p for p in self.net.parameters() if p.requires_grad]
                torch.nn.utils.clip_grad_norm_(params, max_norm=1.0, norm_type=2)

                self.optimizer.step()
                del outputs, logits, l, X, Y, attention_mask
        print("训练结束")

TrainLoRA(net, loss=loss, optimizer=optimizer).train_epochs(num_epochs, train_iter, None)

In [ ]:
torch.save(net.state_dict(), 'SFTwithLoRA.params')

In [ ]:
# 预测
net.eval()

text = f"<|im_start|>user\n播放音源<|im_end|>\n<|im_start|>assistant\n"
tokenizer = "/root/autodl-tmp/SFTwithLoRA/LLM"
tokenizer = AutoTokenizer.from_pretrained(tokenizer, trust_remote_code=True)

inputs = tokenizer(text, return_tensors="pt", truncation=True).to(net.device)
with torch.no_grad():
    outputs = net.generate(**inputs)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
# vLLM
from transformers import AutoTokenizer, AutoModelForCausalLM

state_dict = torch.load("/root/autodl-tmp/SFTwithLoRA/SFTwithLoRA.params")
incompatible_keys = net.load_state_dict(state_dict) 

# net = net.load_state_dict(torch.load("/root/autodl-tmp/SFTwithLoRA/SFTwithLoRA.params", map_location="cuda:0", weights_only=True))
base_model = AutoModelForCausalLM.from_pretrained("/root/autodl-tmp/SFTwithLoRA/LLM", torch_dtype="auto")

def merge_and_restore(model):
    """
    遍历模型，将所有 LoraLayer 合并到原始 Linear 层，并替换回原始 Linear。
    """
    def _merge_recursive(module):
        for name, child in list(module.named_children()):
            if isinstance(child, LoraLayer):
                original_linear = child.original_layer
            else:
                _merge_recursive(child)
        
    _merge_recursive(model)
    return model

model = merge_and_restore(net)

In [ ]:
net